# BPR Financial Data Scraper

Automated scraper untuk mengambil data laporan keuangan BPR dari OJK.

Features:
- Pilihan rentang tahun
- Filter provinsi
- Parallel processing dengan ThreadPoolExecutor
- Progress tracking dengan tqdm

## 1. Import Libraries

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
from tqdm.notebook import tqdm
import time
import re
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=pd.errors.SettingWithCopyWarning)

print("Libraries imported successfully")

Libraries imported successfully


## 2. Load Bank Data

In [2]:
# Contoh data - GANTI dengan load dari CSV Anda
data_banks = {
    'Provinsi': [
        'Provinsi Jawa Barat', 'Provinsi Jawa Barat', 'Provinsi Jawa Barat',
        'Provinsi Jawa Barat', 'Provinsi Jawa Barat', 'Provinsi Papua Barat',
        'Provinsi Papua Barat', 'Provinsi Papua Barat', 'Provinsi Papua Barat',
        'Provinsi Papua Barat'
    ],
    'Kabupaten/Kota': [
        'Kab. Bekasi', 'Kab. Bekasi', 'Kab. Bekasi', 'Kab. Bekasi',
        'Kab. Bekasi', 'Kab. Manokwari', 'Kab. Manokwari', 'Kab. Manokwari',
        'Kota Sorong', 'Kota Sorong'
    ],
    'Nama Bank': [
        'PT Bank Perekonomian Rakyat Cikarang Raharja',
        'PT Bank Perekonomian Rakyat Dana Multi Guna',
        'PT. BPR Siwa Raharja Utama',
        'PT Bank Perekonomian Rakyat Binadana Makmur',
        'PT Bank Perekonomian Rakyat Sentral Mandiri',
        'PT. BPR Arfak Indonesia',
        'PT Bank Perekonomian Rakyat Sinar Mulia Papua',
        'PT BPR Modern Express Papua Barat',
        'PT Bank Perekonomian Rakyat Menara Cendrawasih',
        'PT Bank Perekonomian Rakyat Sorong Sukses Seja'
    ],
    'Kode Bank': [
        '600007', '600036', '600070', '600098', '600103',
        '602643', '602738', '602742', '601289', '602750'
    ]
}

df_banks = pd.DataFrame(data_banks)

# Jika Anda punya file CSV, uncomment dan sesuaikan:
# df_banks = pd.read_csv('your_file.csv')
# Sesuaikan nama kolom jika berbeda:
# df_banks.columns = ['Provinsi', 'Kabupaten/Kota', 'Nama Bank', 'Kode Bank']

df_banks = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/ojk_cfs_bpr_konvensional.csv')
print(f"Total banks loaded: {len(df_banks)}")
print(f"Provinsi available: {df_banks['Provinsi'].unique().tolist()}")
df_banks.head(10)

Total banks loaded: 1863
Provinsi available: ['Provinsi Jawa Barat', 'Provinsi Banten', 'Provinsi DKI Jakarta', 'Provinsi D.I. Yogyakarta', 'Provinsi Jawa Tengah', 'Provinsi Jawa Timur', 'Provinsi Bengkulu', 'Provinsi Jambi', 'Provinsi NAD', 'Provinsi Sumatera Utara', 'Provinsi Sumatera Barat', 'Provinsi Riau', 'Provinsi Sumatera Selatan', 'Provinsi Kep. Bangka Belitung', 'Provinsi Kep. Riau', 'Provinsi Lampung', 'Provinsi Kalimantan Selatan', 'Provinsi Kalimantan Barat', 'Provinsi Kalimantan Timur', 'Provinsi Kalimantan Tengah', 'Provinsi Sulawesi Tengah', 'Provinsi Sulawesi Selatan', 'Provinsi Sulawesi Utara', 'Provinsi Gorontalo', 'Provinsi Sulawesi Barat', 'Provinsi Sulawesi Tenggara', 'Provinsi Nusa Tenggara Barat', 'Provinsi Bali', 'Provinsi Nusa Tenggara Timur', 'Provinsi Maluku', 'Provinsi Papua', 'Provinsi Maluku Utara', 'Provinsi Papua Barat']


,Provinsi,Kabupaten/Kota,Nama Bank,Kode Bank
0,Provinsi Jawa Barat,Kab. Bekasi,PT Bank Perekonomian Rakyat Cikarang Raharja,600007
1,Provinsi Jawa Barat,Kab. Bekasi,PT Bank Perekonomian Rakyat Dana Multi Guna,600036
2,Provinsi Jawa Barat,Kab. Bekasi,PT. BPR Siwa Raharja Utama,600070
3,Provinsi Jawa Barat,Kab. Bekasi,PT Bank Perekonomian Rakyat Binadana Makmur,600098
4,Provinsi Jawa Barat,Kab. Bekasi,PT Bank Perekonomian Rakyat Sentral Mandiri,600103
5,Provinsi Jawa Barat,Kab. Bekasi,PT Bank Perekonomian Rakyat Antar Guna,600108
6,Provinsi Jawa Barat,Kab. Bekasi,PT Bank Perekonomian Rakyat Artha Sentana Hardja,600131
7,Provinsi Jawa Barat,Kab. Bekasi,PT Bank Perekonomian Rakyat Karya Kurniautama,600135
8,Provinsi Jawa Barat,Kab. Bekasi,PT. BPR Artaprima Danajasa,600137
9,Provinsi Jawa Barat,Kab. Bekasi,PT Bank Perekonomian Rakyat Olympindo Primadana,600139


## 3. Configuration

In [3]:
BASE_URL = "https://cfs.ojk.go.id/cfs/ReportViewerForm.aspx"
MONTH = "12"  # Fixed: Desember
PERIOD_TYPE = "R"

REPORT_TYPES = {
    "BPK-901-000001": "Laporan Posisi Keuangan",
    "BPK-901-000002": "Laporan Laba Rugi",
    "BPK-901-000003": "Laporan Kualitas Aset Produktif"
}

COLUMN_ORDER = [
    'Tahun', 'Bulan', 'Longitude', 'Latitude', 'Nama_BPR', 
    'Kabupaten_Kota', 'Provinsi', 'Kode_Bank',
    # Posisi Keuangan
    'Total_Aset', 
    'a. Kepada BPR',
    'b. Kepada Bank Umum',
    'c. Kepada non bank – pihak terkait',
    'd. Kepada non bank – pihak tidak terkait',
    'Kredit_Kepada_BPR', 
    'Kredit_Kepada_Bank_Umum',
    'Kredit_Pihak_Terkait', 
    'Kredit_Pihak_Tidak_Terkait', 
    'Cadangan_Kerugian_Penurunan_Nilai',
    'Jumlah_Kredit',
    'Tabungan', 
    'Deposito', 
    'Total_Liabilitas', 
    'Laba_Tahun_Berjalan', 
    'Total_Ekuitas',
    'Agunan_Yang_Diambil_Alih',
    # Laba Rugi
    'Jumlah_Pendapatan_Bunga', 
    'Beban_Kerugian_Penurunan_Nilai',
    'Jumlah_Pendapatan_Operasional', 
    'Jumlah_Beban_Operasional',
    'Laba_Rugi_Berjalan',
    # Rasio Keuangan
    'NPL_Neto', 
    'NPL_Gross', 
    'ROA', 
    'BOPO', 
    'NIM', 
    'LDR', 
    'KPMM',
    'Cash_Ratio'
]

print("Configuration loaded")

Configuration loaded


## 4. Helper Functions

In [4]:
def clean_number(value):
    """Convert string number dengan format Indonesia ke float"""
    if pd.isna(value) or value == '' or value == 'NaN':
        return 0
    
    if isinstance(value, (int, float)):
        return float(value)
    
    value = str(value).strip()
    value = re.sub(r'[^\d.,()\-]', '', value)
    
    if '(' in value and ')' in value:
        value = '-' + value.replace('(', '').replace(')', '')
    
    value = value.replace(',', '')
    
    try:
        return float(value)
    except:
        return 0

def find_main_table(tables):
    """Cari tabel utama yang berisi data laporan"""
    best_table = None
    max_score = 0
    best_idx = -1
    
    for idx, table in enumerate(tables):
        try:
            html_string = str(table)
            df = pd.read_html(StringIO(html_string))[0]
            
            score = 0
            if df.shape[0] > 20:
                score += df.shape[0]
            if 2 <= df.shape[1] <= 10:
                score += 50
            
            numeric_count = 0
            for col in df.columns:
                try:
                    nums = pd.to_numeric(df[col], errors='coerce')
                    if (nums > 1000).sum() > 5:
                        numeric_count += 1
                except:
                    pass
            
            if numeric_count > 0:
                score += numeric_count * 30
            
            if score > max_score:
                max_score = score
                best_table = df
                best_idx = idx
                
        except Exception as e:
            continue
    
    return best_table, best_idx

def find_value_in_row(df, keywords, col_index=2):
    """Cari nilai di tabel berdasarkan keyword"""
    for col in df.columns[:2]:
        for keyword in keywords:
            mask = df[col].astype(str).str.contains(keyword, case=False, regex=False, na=False)
            if mask.any():
                idx = df[mask].index[0]
                if col_index < len(df.columns):
                    return clean_number(df.iloc[idx, col_index])
                return 0
    return 0

print("Helper functions defined")

Helper functions defined


## 5. Extraction & Parsing Functions

In [5]:
def extract_table(bank_code_number, bank_code, year, report_type, timeout=15):
    """Ekstrak tabel dari laporan"""
    params = {
        'BankCodeNumber': bank_code_number,
        'BankCode': bank_code,
        'Month': MONTH,
        'Year': str(year),
        'FinancialReportPeriodTypeCode': PERIOD_TYPE,
        'FinancialReportTypeCode': report_type
    }
    
    try:
        response = requests.get(BASE_URL, params=params, timeout=timeout)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        tables = soup.find_all('table')
        
        if not tables:
            return None
        
        main_table, idx = find_main_table(tables)
        return main_table
        
    except Exception as e:
        return None

def parse_posisi_keuangan(df):
    """Parse Laporan Posisi Keuangan"""
    data = {}
    
    # Fields mapping untuk nilai yang ada di kolom utama
    fields_mapping = {
        'Total_Aset': [['Total Aset'], 2],
        'Kredit_Kepada_BPR': [['a. Kepada BPR', 'Kepada BPR'], 2],
        'Kredit_Kepada_Bank_Umum': [['b. Kepada Bank Umum', 'Kepada Bank Umum'], 2],
        'Kredit_Pihak_Terkait': [['c. Kepada non bank – pihak terkait'], 2],
        'Kredit_Pihak_Tidak_Terkait': [['d. Kepada non bank – pihak tidak terkait'], 2],
        'Tabungan': [['a. Tabungan', 'a.    Tabungan'], 2],
        'Deposito': [['b. Deposito', 'b.    Deposito'], 2],
        'Total_Liabilitas': [['Total Liabilitas'], 2],
        'Agunan_Yang_Diambil_Alih': [['Agunan yang Diambil Alih'], 2],
        'Laba_Tahun_Berjalan': [['b. Tahun Berjalan'], 2],
        'Total_Ekuitas': [['Total Ekuitas'], 2],
        'a. Kepada BPR': [['a. Kepada BPR', 'Kepada BPR'], 2],
        'b. Kepada Bank Umum': [['b. Kepada Bank Umum', 'Kepada Bank Umum'], 2],
        'c. Kepada non bank – pihak terkait': [['c. Kepada non bank – pihak terkait'], 2],
        'd. Kepada non bank – pihak tidak terkait': [['d. Kepada non bank – pihak tidak terkait'], 2],
    }
    
    for field, (keywords, col_idx) in fields_mapping.items():
        value = find_value_in_row(df, keywords, col_idx)
        data[field] = value
    
    # Special handling untuk Cadangan_Kerugian_Penurunan_Nilai dan Jumlah_Kredit
    # Cari baris "Kredit yang Diberikan" dulu
    kredit_section_found = False
    start_idx = None
    
    for col in df.columns[:2]:
        mask = df[col].astype(str).str.contains('Kredit yang Diberikan', case=False, regex=False, na=False)
        if mask.any():
            start_idx = df[mask].index[0]
            kredit_section_found = True
            break
    
    if kredit_section_found and start_idx is not None:
        # Ambil subset dataframe mulai dari "Kredit yang Diberikan"
        # Biasanya data yang kita cari ada dalam 10-15 baris ke bawah
        subset_df = df.iloc[start_idx:start_idx+15].copy()
        
        # Cari Cadangan Kerugian Penurunan Nilai
        for col in subset_df.columns[:2]:
            mask = subset_df[col].astype(str).str.contains(
                '-/- Cadangan Kerugian Penurunan Nilai|-/- Penyisihan Penghapusan Aset Produktif', 
                case=False, regex=True, na=False
            )
            if mask.any():
                idx = subset_df[mask].index[0]
                if 2 < len(subset_df.columns):
                    data['Cadangan_Kerugian_Penurunan_Nilai'] = clean_number(subset_df.iloc[subset_df.index.get_loc(idx), 2])
                break
        
        # Cari Jumlah (total kredit yang diberikan)
        for col in subset_df.columns[:2]:
            mask = subset_df[col].astype(str).str.match(r'^Jumlah$', case=False, na=False)
            if mask.any():
                idx = subset_df[mask].index[0]
                if 2 < len(subset_df.columns):
                    data['Jumlah_Kredit'] = clean_number(subset_df.iloc[subset_df.index.get_loc(idx), 2])
                break
    
    # Jika tidak ketemu dengan cara di atas, gunakan fallback calculation
    if 'Cadangan_Kerugian_Penurunan_Nilai' not in data:
        data['Cadangan_Kerugian_Penurunan_Nilai'] = 0
    
    if 'Jumlah_Kredit' not in data:
        data['Jumlah_Kredit'] = (
            data.get('Kredit_Kepada_BPR', 0) + 
            data.get('Kredit_Kepada_Bank_Umum', 0) + 
            data.get('Kredit_Pihak_Terkait', 0) + 
            data.get('Kredit_Pihak_Tidak_Terkait', 0)
        )
    
    return data

def parse_laba_rugi(df):
    """Parse Laporan Laba Rugi"""
    data = {}
    
    fields_mapping = {
        'Jumlah_Pendapatan_Bunga': [['Jumlah Pendapatan Bunga'], 2],
        'Beban_Kerugian_Penurunan_Nilai': [['Beban Kerugian Penurunan Nilai'], 2],
        'Jumlah_Pendapatan_Operasional': [['JUMLAH PENDAPATAN OPERASIONAL'], 2],
        'Jumlah_Beban_Operasional': [['JUMLAH BEBAN OPERASIONAL'], 2],
        'Laba_Rugi_Berjalan': [['LABA (RUGI) TAHUN BERJALAN SEBELUM PAJAK PENGHASILAN'], 2],
        'Jumlah_Pendapatan_Bunga': [['Jumlah Pendapatan Bunga'], 2],
        'Beban_Kerugian_Penurunan_Nilai': [['Beban Kerugian Penurunan Nilai'], 2],
    }
    
    for field, (keywords, col_idx) in fields_mapping.items():
        value = find_value_in_row(df, keywords, col_idx)
        data[field] = value
    
    return data

def parse_kualitas_aset(df):
    """Parse Laporan Kualitas Aset Produktif"""
    data = {}
    
    rasio_keywords = {
        'NPL_Neto': ['Non Performing Loan (NPL) Neto', '(NPL) Neto', 'NPL (neto)'],
        'NPL_Gross': ['Non Performing Loan (NPL) Gross', 'NPL) Gross'],
        'ROA': ['Return on Assets (ROA)', '(ROA)', 'ROA'],
        'BOPO': ['Biaya Operasional terhadap Pendapatan Operasional (BOPO)', '(BOPO)', 'BOPO'],
        'NIM': ['Net Interest Margin (NIM)', '(NIM)'],
        'LDR': ['Loan to Deposit Ratio (LDR)', '(LDR)', 'LDR'],
        'KPMM': ['Kewajiban Penyediaan Modal Minimum (KPMM)', '(KPMM)', 'KPMM'],
        'Cash_Ratio': ['Cash Ratio', 'Cash Ratio']
    }
    
    for field, keywords in rasio_keywords.items():
        found = False
        for col_idx, col in enumerate(df.columns):
            for keyword in keywords:
                mask = df[col].astype(str).str.contains(keyword, case=False, regex=False, na=False)
                if mask.any():
                    idx = df[mask].index[0]
                    for next_col_idx in range(col_idx + 1, len(df.columns)):
                        val = df.iloc[idx, next_col_idx]
                        if pd.notna(val) and str(val).strip() != '':
                            data[field] = clean_number(val)
                            found = True
                            break
                    if found:
                        break
            if found:
                break
        
        if field not in data:
            data[field] = 0
    
    return data

print("Parsing functions defined")

Parsing functions defined


## 6. Main Scraper Class

In [6]:
class BPRScraper:
    """Main scraper class untuk handle scraping dengan thread pool"""
    
    def __init__(self):
        self.results = []
        
    def scrape_single_bank(self, bank_row, year):
        """Scrape data untuk satu bank di satu tahun"""
        bank_code_number = str(bank_row['Kode Bank'])
        bank_code = bank_row['Nama Bank']
        
        result = {
            'Tahun': int(year),
            'Bulan': MONTH,
            'Nama_BPR': bank_code,
            'Kabupaten_Kota': bank_row['Kabupaten/Kota'],
            'Provinsi': bank_row['Provinsi'],
            'Kode_Bank': bank_code_number,
            'Longitude': None,
            'Latitude': None,
            'status': 'failed'
        }
        
        try:
            # Laporan Posisi Keuangan
            df_posisi = extract_table(bank_code_number, bank_code, year, "BPK-901-000001")
            if df_posisi is not None:
                posisi_data = parse_posisi_keuangan(df_posisi)
                result.update(posisi_data)
            
            time.sleep(0.1)
            
            # Laporan Laba Rugi
            df_laba = extract_table(bank_code_number, bank_code, year, "BPK-901-000002")
            if df_laba is not None:
                laba_data = parse_laba_rugi(df_laba)
                result.update(laba_data)
            
            time.sleep(0.1)
            
            # Laporan Kualitas Aset
            df_kualitas = extract_table(bank_code_number, bank_code, year, "BPK-901-000003")
            if df_kualitas is not None:
                kualitas_data = parse_kualitas_aset(df_kualitas)
                result.update(kualitas_data)
            
            result['status'] = 'success'
            
        except Exception as e:
            result['error'] = str(e)
        
        return result
    
    def run(self, df_banks_filtered, years, max_workers=4):
        """Run scraping dengan ThreadPoolExecutor"""
        tasks = []
        for _, bank_row in df_banks_filtered.iterrows():
            for year in years:
                tasks.append((bank_row, year))
        
        total_tasks = len(tasks)
        results = []
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(self.scrape_single_bank, task[0], task[1]): task for task in tasks}
            
            with tqdm(total=total_tasks, desc="Scraping", unit="task") as pbar:
                for future in as_completed(futures):
                    try:
                        result = future.result()
                        results.append(result)
                    except Exception as e:
                        print(f"Task failed: {e}")
                    pbar.update(1)
        
        return results

print("BPRScraper class defined")

BPRScraper class defined


## 7. USER CONFIGURATION

Edit parameter di bawah ini sesuai kebutuhan:

In [15]:
# ============================================================================
# EDIT PARAMETER DI SINI
# ============================================================================

# Rentang tahun (Fixed: Desember)
YEAR_START = 2020
YEAR_END = 2024

# Pilih provinsi:
# - ['SEMUA'] untuk scrape semua provinsi
# - Atau list provinsi spesifik, contoh: ['Provinsi Jawa Barat', 'Provinsi Papua Barat']
SELECTED_PROVINCES = ['SEMUA']

# Jumlah thread workers (10-15 workers = optimal untuk speed)
MAX_WORKERS = 8

# Nama file output (tanpa extension)
OUTPUT_FILENAME = 'bpr_financial_data'

# ============================================================================

print("Configuration set:")
print(f"  Tahun: {YEAR_START} - {YEAR_END}")
print(f"  Provinsi: {SELECTED_PROVINCES}")
print(f"  Workers: {MAX_WORKERS}")
print(f"  Output: {OUTPUT_FILENAME}")

Configuration set:
  Tahun: 2020 - 2024
  Provinsi: ['SEMUA']
  Workers: 8
  Output: bpr_financial_data


## 8. RUN SCRAPING

Jalankan cell ini untuk memulai scraping:

In [16]:
# Prepare data
years = list(range(YEAR_START, YEAR_END + 1))

# Filter banks by province
if 'SEMUA' in SELECTED_PROVINCES:
    filtered_banks = df_banks.copy()
else:
    filtered_banks = df_banks[df_banks['Provinsi'].isin(SELECTED_PROVINCES)].copy()

total_tasks = len(filtered_banks) * len(years)

print("="*70)
print("STARTING SCRAPING")
print("="*70)
print(f"Tahun: {YEAR_START} - {YEAR_END} ({len(years)} tahun)")
print(f"Bulan: Desember (Fixed)")
print(f"Provinsi: {', '.join(SELECTED_PROVINCES)}")
print(f"Jumlah Bank: {len(filtered_banks)}")
print(f"Total Tasks: {total_tasks}")
print(f"Workers: {MAX_WORKERS}")
print("="*70)
print()

# Run scraping
start_time = time.time()
scraper = BPRScraper()
results = scraper.run(filtered_banks, years, max_workers=MAX_WORKERS)
elapsed_time = time.time() - start_time

# Process results
df_results = pd.DataFrame(results)

# Add missing columns
for col in COLUMN_ORDER:
    if col not in df_results.columns:
        df_results[col] = 0

# Reorder columns
available_cols = [col for col in COLUMN_ORDER if col in df_results.columns]
df_results = df_results[available_cols]

# Statistics
success_count = (df_results['status'] == 'success').sum() if 'status' in df_results.columns else len(df_results)
failed_count = total_tasks - success_count

print("\n" + "="*70)
print("SCRAPING COMPLETED")
print("="*70)
print(f"Total Tasks: {total_tasks}")
print(f"Success: {success_count} ({success_count/total_tasks*100:.1f}%)")
print(f"Failed: {failed_count} ({failed_count/total_tasks*100:.1f}%)")
print(f"Elapsed Time: {elapsed_time:.2f} seconds")
print(f"Average Time per Task: {elapsed_time/total_tasks:.2f} seconds")
print("="*70)

# Remove status column
df_export = df_results.drop(columns=['status'], errors='ignore')

# Save files
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_file = f"{OUTPUT_FILENAME}_{timestamp}.csv"
df_export.to_csv(csv_file, index=False, encoding='utf-8-sig')
print(f"\nSaved: {csv_file}")

try:
    excel_file = f"{OUTPUT_FILENAME}_{timestamp}.xlsx"
    df_export.to_excel(excel_file, index=False, engine='openpyxl')
    print(f"Saved: {excel_file}")
except Exception as e:
    print(f"Warning: Could not save Excel: {e}")

print("\nDone!")

STARTING SCRAPING
Tahun: 2020 - 2024 (5 tahun)
Bulan: Desember (Fixed)
Provinsi: SEMUA
Jumlah Bank: 1863
Total Tasks: 9315
Workers: 8



Scraping:   0%|          | 0/9315 [00:00<?, ?task/s]


SCRAPING COMPLETED
Total Tasks: 9315
Success: 9315 (100.0%)
Failed: 0 (0.0%)
Elapsed Time: 22174.46 seconds
Average Time per Task: 2.38 seconds

Saved: bpr_financial_data_20260101_034824.csv
Saved: bpr_financial_data_20260101_034824.xlsx

Done!


## 9. Preview Results

In [17]:
print("Sample Data (First 5 rows):")
df_export.head()

Sample Data (First 5 rows):


,Tahun,Bulan,Longitude,Latitude,Nama_BPR,Kabupaten_Kota,Provinsi,Kode_Bank,Total_Aset,a. Kepada BPR,...,Jumlah_Beban_Operasional,Laba_Rugi_Berjalan,NPL_Neto,NPL_Gross,ROA,BOPO,NIM,LDR,KPMM,Cash_Ratio
0,2021,12,None,None,PT Bank Perekonomian Rakyat Dana Multi Guna,Kab. Bekasi,Provinsi Jawa Barat,600036,26196479.0,0.0,...,4885829.0,723363.0,15.53,0.00,2.76,87.90,0.00,67.28,46.94,10.76
1,2022,12,None,None,PT Bank Perekonomian Rakyat Cikarang Raharja,Kab. Bekasi,Provinsi Jawa Barat,600007,98643979.0,0.0,...,10298809.0,15225.0,30.50,0.00,0.02,99.58,0.00,55.52,15.62,23.30
2,2021,12,None,None,PT Bank Perekonomian Rakyat Cikarang Raharja,Kab. Bekasi,Provinsi Jawa Barat,600007,89311789.0,0.0,...,12048856.0,-3799468.0,38.13,0.00,-4.25,149.36,0.00,57.80,12.87,26.50
3,2023,12,None,None,PT Bank Perekonomian Rakyat Cikarang Raharja,Kab. Bekasi,Provinsi Jawa Barat,600007,58263419.0,0.0,...,19703655.0,-10439347.0,23.97,38.66,0.00,205.96,4.68,42.81,14.63,32.90
4,2020,12,None,None,PT Bank Perekonomian Rakyat Cikarang Raharja,Kab. Bekasi,Provinsi Jawa Barat,600007,93117374.0,0.0,...,14306270.0,-1623346.0,38.00,0.00,-1.74,112.63,0.00,57.81,13.11,11.39


In [18]:
df_export.columns

Index(['Tahun', 'Bulan', 'Longitude', 'Latitude', 'Nama_BPR', 'Kabupaten_Kota',
       'Provinsi', 'Kode_Bank', 'Total_Aset', 'a. Kepada BPR',
       'b. Kepada Bank Umum', 'c. Kepada non bank – pihak terkait',
       'd. Kepada non bank – pihak tidak terkait', 'Kredit_Kepada_BPR',
       'Kredit_Kepada_Bank_Umum', 'Kredit_Pihak_Terkait',
       'Kredit_Pihak_Tidak_Terkait', 'Cadangan_Kerugian_Penurunan_Nilai',
       'Jumlah_Kredit', 'Tabungan', 'Deposito', 'Total_Liabilitas',
       'Laba_Tahun_Berjalan', 'Total_Ekuitas', 'Agunan_Yang_Diambil_Alih',
       'Jumlah_Pendapatan_Bunga', 'Beban_Kerugian_Penurunan_Nilai',
       'Jumlah_Pendapatan_Operasional', 'Jumlah_Beban_Operasional',
       'Laba_Rugi_Berjalan', 'NPL_Neto', 'NPL_Gross', 'ROA', 'BOPO', 'NIM',
       'LDR', 'KPMM', 'Cash_Ratio'],
      dtype='object')

In [19]:
print("Summary Statistics:")
df_export.describe()

Summary Statistics:


,Tahun,Total_Aset,a. Kepada BPR,b. Kepada Bank Umum,c. Kepada non bank – pihak terkait,d. Kepada non bank – pihak tidak terkait,Kredit_Kepada_BPR,Kredit_Kepada_Bank_Umum,Kredit_Pihak_Terkait,Kredit_Pihak_Tidak_Terkait,...,Jumlah_Beban_Operasional,Laba_Rugi_Berjalan,NPL_Neto,NPL_Gross,ROA,BOPO,NIM,LDR,KPMM,Cash_Ratio
count,9315.000000,7.069000e+03,7.069000e+03,7069.0,7.069000e+03,7.069000e+03,7.069000e+03,7069.0,7.069000e+03,7.069000e+03,...,7.122000e+03,7.122000e+03,7126.000000,7126.000000,7126.000000,7126.000000,7126.000000,7126.000000,7126.000000,7126.000000
mean,2022.000000,2.896519e+10,1.695032e+08,0.0,4.245207e+08,2.050409e+10,1.695032e+08,0.0,4.245207e+08,2.050409e+10,...,3.350048e+09,5.465610e+08,9.689063,5.326103,1.032727,98.801207,4.561353,92.959893,52.802457,42.871988
std,1.414289,2.164367e+11,5.320532e+09,0.0,4.782606e+09,1.672731e+11,5.320532e+09,0.0,4.782606e+09,1.672731e+11,...,2.035945e+10,6.534990e+09,27.655202,10.244601,23.068201,161.828417,11.369273,179.670313,111.045813,279.518501
min,2020.000000,5.577000e+03,0.000000e+00,0.0,-8.300000e+01,0.000000e+00,0.000000e+00,0.0,-8.300000e+01,0.000000e+00,...,0.000000e+00,-9.889262e+10,0.000000,0.000000,-999.990000,0.000000,-10.350000,0.000000,-769.320000,0.000000
25%,2021.000000,2.556899e+07,0.000000e+00,0.0,6.843300e+04,1.637809e+07,0.000000e+00,0.0,6.843300e+04,1.637809e+07,...,3.805823e+06,1.805945e+05,3.060000,0.000000,0.570000,78.852500,0.000000,67.282500,25.145000,13.270000
50%,2022.000000,6.165410e+07,0.000000e+00,0.0,4.030920e+05,4.161214e+07,0.000000e+00,0.0,4.030920e+05,4.161214e+07,...,8.400415e+06,1.111521e+06,6.420000,0.000000,2.010000,86.880000,0.000000,79.340000,37.760000,20.240000
75%,2023.000000,3.414169e+08,0.000000e+00,0.0,2.111665e+06,2.220980e+08,0.000000e+00,0.0,2.111665e+06,2.220980e+08,...,3.674333e+07,5.321616e+06,12.520000,7.445000,3.640000,96.420000,8.260000,90.930000,60.507500,31.275000
max,2024.000000,1.120546e+13,2.855531e+11,0.0,2.091464e+11,8.964319e+12,2.855531e+11,0.0,2.091464e+11,8.964319e+12,...,8.284465e+11,3.581006e+11,1559.000000,100.000000,489.000000,8584.000000,565.880000,7551.000000,7252.460000,9999.870000


## 10. Data Summary by Province

In [12]:
print("Data Count by Province:")
df_export.groupby('Provinsi').size().sort_values(ascending=False)

Data Count by Province:


Provinsi
Provinsi Jawa Barat              464
Provinsi Jawa Timur              324
Provinsi Jawa Tengah             258
Provinsi Bali                    137
Provinsi Sumatera Barat           97
Provinsi Banten                   71
Provinsi Sumatera Utara           56
Provinsi D.I. Yogyakarta          55
Provinsi Kep. Riau                44
Provinsi DKI Jakarta              32
Provinsi Riau                     32
Provinsi Nusa Tenggara Barat      30
Provinsi Lampung                  28
Provinsi Kalimantan Selatan       28
Provinsi Sumatera Selatan         24
Provinsi Kalimantan Barat         22
Provinsi Sulawesi Selatan         22
Provinsi Jambi                    20
Provinsi Sulawesi Utara           18
Provinsi Sulawesi Tenggara        18
Provinsi Kalimantan Timur         16
Provinsi Nusa Tenggara Timur      12
Provinsi Papua                     9
Provinsi Sulawesi Tengah           9
Provinsi Kalimantan Tengah         6
Provinsi NAD                       5
Provinsi Maluku Utara        

## 11. Top Banks by Total Aset

In [13]:
# Get latest year data
latest_year = df_export['Tahun'].max()
df_latest = df_export[df_export['Tahun'] == latest_year].copy()

print(f"Top 10 Banks by Total Aset ({latest_year}):")
df_top = df_latest.nlargest(10, 'Total_Aset')[['Nama_BPR', 'Provinsi', 'Total_Aset', 'ROA', 'NPL_Gross']]
df_top

Top 10 Banks by Total Aset (2019):


,Nama_BPR,Provinsi,Total_Aset,ROA,NPL_Gross
1519,PT Bank Perekonomian Rakyat Eka Bumi Artha,Provinsi Lampung,8.482554e+09,3.45,0.0
1817,PT Bank Perekonomian Rakyat Lestari Bali,Provinsi Bali,6.186800e+09,3.11,0.0
728,PT Bank Perekonomian Rakyat Surya Yudhakencana,Provinsi Jawa Tengah,2.864344e+09,3.25,0.0
343,PT Bank Perekonomian Rakyat Karyajatnika Sadaya,Provinsi Jawa Barat,2.571387e+09,0.11,0.0
1162,PT Bank Perekonomian Rakyat Jawa Timur Perseroda,Provinsi Jawa Timur,2.565632e+09,1.04,0.0
1618,PT Bank Perekonomian Rakyat Hasamitra,Provinsi Sulawesi Selatan,2.435711e+09,2.33,0.0
1595,PT. BPR Palu Lokadana Utama,Provinsi Sulawesi Tengah,2.310746e+09,4.46,0.0
1845,PT Bank Perekonomian Rakyat Modern Express,Provinsi Maluku,2.121520e+09,4.95,0.0
1468,PT Bank Perekonomian Rakyat Dana Nusantara,Provinsi Kep. Riau,1.681911e+09,3.40,0.0
1515,PT Bank Perekonomian Rakyat Utomo Manunggal Se...,Provinsi Lampung,1.539662e+09,4.40,0.0


In [20]:
data = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/bpr_financial_data_20260101_034824.csv')
df = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/bpr_financial_data_20251231_212650.csv')

In [29]:
dx = pd.concat([data, df], ignore_index=True)
dx = dx.sort_values(by=['Tahun'], ascending=[True])
dx = dx.reset_index(drop=True)
dx.to_csv('Full1924.csv', index=False)

In [38]:
dx.head()

,Tahun,Bulan,Longitude,Latitude,Nama_BPR,Kabupaten_Kota,Provinsi,Kode_Bank,Total_Aset,a. Kepada BPR,...,Jumlah_Beban_Operasional,Laba_Rugi_Berjalan,NPL_Neto,NPL_Gross,ROA,BOPO,NIM,LDR,KPMM,Cash_Ratio
0,2019,12,NaN,NaN,PT BPR Modern Express Papua Barat,Kab. Manokwari,Provinsi Papua Barat,602742,65658924.0,0.0,...,8310972.0,1009266.0,0.26,0.0,1.86,92.19,0.0,90.23,16.73,14.87
1,2019,12,NaN,NaN,PT Bank Perekonomian Rakyat Mitra Pati Mandiri,Kab. Pati,Provinsi Jawa Tengah,601441,40360979.0,0.0,...,7169079.0,1391922.0,3.25,0.0,2.93,83.64,0.0,79.19,23.44,25.96
2,2019,12,NaN,NaN,PT Bank Perekonomian Rakyat Asabahana Sejahtera,Kab. Pati,Provinsi Jawa Tengah,601458,21306442.0,0.0,...,3568365.0,793456.0,12.88,0.0,4.08,81.17,0.0,86.83,51.45,11.42
3,2019,12,NaN,NaN,PT. BPR Juwana Artha Sentosa,Kab. Pati,Provinsi Jawa Tengah,601460,58190823.0,0.0,...,9412098.0,675935.0,12.94,0.0,1.19,92.65,0.0,90.64,17.05,10.45
4,2019,12,NaN,NaN,PT BPR BKK Pati ( Perseroda ),Kab. Pati,Provinsi Jawa Tengah,601507,313030941.0,0.0,...,38543872.0,6977761.0,5.47,0.0,2.40,84.17,0.0,68.18,26.50,13.90


---

## Notes:

### Perbedaan dari Versi Sebelumnya:
- Menggunakan `ThreadPoolExecutor` instead of `multiprocessing.Pool`
- Lebih compatible dengan Jupyter Notebook
- Tidak ada masalah pickle error
- Thread-safe dan reliable

### Cara Pakai:
1. Run semua cell dari atas ke bawah
2. Edit parameter di Section 7
3. Run Section 8 untuk start scraping
4. File output otomatis tersimpan

### Tips:
- Start dengan MAX_WORKERS=4, naikan bertahap jika stabil
- Untuk dataset besar, scrape per provinsi
- Monitor memory usage untuk dataset sangat besar
- Jika terlalu banyak timeout, kurangi MAX_WORKERS